In [ ]:
import pandas as pd
import geopandas as gpd
import chardet
import math as m
from math import pi
from scipy import stats
import csv
import matplotlib.pyplot as plt
from tkinter import  filedialog as filedialog

## Funções utilizadas

In [158]:
def leitor_de_arquivo(caminho, extensao):
    excel = ['xls', 'xlsx', 'xlsm', 'xlsb', 'odf', 'ods']

    if extensao in excel:
        # pd.ExcelFile(arquivo).sheet_names
        df = pd.read_excel(caminho, sheet_name = 1)
        return df
    
    elif extensao == 'csv':
        sep = [',', ';', '|', ' ']
        for s in sep:
            try:
                df = pd.read_csv(caminho,
                                sep = s,
                                encoding = 'latin-1')
                if len(df.columns) > 1:
                    return df
                else: 
                    print(f'ERRO com sep "{s}"')
            except Exception as e:
                print(e)
    else:
        print('Arquivo não suportado')

def padronizar_crs(shp1, shp2):
    if shp1.crs == shp2.crs:
        print('Ambos os arquivos possuem o mesmo sistema de coordenadas')
    else:
        shp1.to_crs(shp2.crs, inplace=True)
        print(f'O CRS do shapefile 1 foi alterado para o mesmo CRS do shapefile 2!')
        if shp1.crs == shp2.crs:
            print(shp1.crs.name)


def obter_extensao(caminho):
    return caminho.split('.')[1]

def resultados_simples(biomassa_t_ha,std_da_media, n, intervalo_de_confianca, erro_amostral, erro_padrao, area_total):
    print('--- Resultados ---')
    print(f'Estoque de biomassa: {biomassa_t_ha:.2f} Mt/ha')
    print(f'Área total: {area_total:.2f}')
    print(f'Estoque de biomassa em área total: {(biomassa_t_ha * area_total):,.2f} Mg')
    print(f'Desvio padrão da média: {std_da_media:.2f}')
    print(f'Indivíduos avaliados: {n}')
    print(f'IC (95%): +/- {intervalo_de_confianca:.2f} Mg/ha')
    print(f'Erro padrão: {erro_padrao:.2f} Mg/ha')
    print(f'Erro amostral: {erro_amostral:.1f}%\n')

# ------------- Equações alométricas -------------
# R² = 0.7005
def eq_biomassa_jamacaru (dap):
    return 0.0010 * pow(dap, 3.2327)

# R² = 0.9184
def eq_geral_simples_entrada (area_basal):
    return 0.2283 * pow(area_basal, 1.1475)
  
# R² = 0.9494
def eq_geral_dupla_entrada (area_basal, altura_m):
    return 0.1085 * pow((area_basal * altura_m), 0.9497)

def eq_small_spp (dap):
    return  0.2627 * pow(dap, 1.901)

def eq_large_spp (dap):
    return 0.2368 * pow(dap, 2.2219)

### Carregando dados

In [ ]:
# pd.read_excel(r'C:/Users/Bruno/Desktop/Python/BrCarbon/avaliaçao_bruno/data/raw/dados_inventario.xlsx')

# Abre um modal para seleção do arquivo como caminho (string)
caminho_dataset = filedialog.askopenfilename(title="DATASET")
caminho_parcelas = filedialog.askopenfilename(title="PARCELAS")
caminho_estratos = filedialog.askopenfilename(title="ESTRATOS")

# Split separa a str de acordo com o separador definido e retorna uma lista 
extensao_arquivo = obter_extensao(caminho_dataset)

# Função otimizada que evita erros, principalmente com separadores e encoding de csv
dados = leitor_de_arquivo(caminho_dataset, extensao_arquivo)
dados.head()

Uma vez que a análise por estrato é necessária para o exercício, e também para que não tenhamos uma estimativa superestimada do estoque de biomassa, farei um spatial join no início do processamento para que o dataset contenha uma coluna indicativa do estrato.<br><br>Antes do join, precisamos ter certeza que ambas geometrias estão no mesmos Sistema Coordenadas (CRS).<br><br>As principais informações que precisamos extrair dos shapefiles são:
- Área total
- Área de cada estrato
- Quais parcelas pertencem a cada estrato

1. Lendo os arquivos e padronizando CRS

In [78]:
parcelas = gpd.read_file(caminho_parcelas)
estratos = gpd.read_file(caminho_estratos)

padronizar_crs(parcelas, estratos)

O CRS do shapefile 1 foi alterado para o mesmo CRS do shapefile 2!
WGS 84 / UTM zone 24S


Unindo os GeoDataFrames para extrair área total e por estrato. Usaremos mais a frente para calcular as estimativas por estrato e área total. 

In [82]:
# O primeiro shp é o que mantêm as geometrias no join
parcelas_estratos = gpd.sjoin(parcelas, estratos, how='inner') # within > parcelas within estratos

parcelas_estratos = parcelas_estratos[['ID', 'grupo']]

Por padrão, a unidade de medida do atributo `.area` é a mesma do CRS. Nesse caso, o CRS WGS 84 está em metros, então o atributo `.area` retorna m² como unidade de medida.

In [ ]:
# 0 = Cristalino
# 1 = Sedimentar

# area do estrato / 10k para transformar em m²
cristalino_area_ha = estratos.area[0] / 10_000 
sedimentar_area_ha = estratos.area[1] / 10_000
area_total_ha = cristalino_area_ha + sedimentar_area_ha

6193.341690950096


Para sabermos quais parcelas estão em cada estrato, fazemos um `merge` das geometrias com o dataset utilizando como ID comum a coluna ID e, em seguida, renomeado-a para facilitar a associação com "parcela".

In [97]:
df = pd.merge(dados, parcelas_estratos, how='inner', on='ID')

df.rename(columns = {'ID': 'ID_parcela', 'grupo': 'GRUPO'}, 
                     inplace = True) # inplace evita criar outro df

# Explorando os dados

Aqui a ideia é ter uma noção geral dos dados, se tem algum dado faltando ou que foi preenchido de forma errada (acidentalemente incluir um 0 a mais ou a menos) e distribuição geral das variáveis. <br><br> No artigo, o autor descreve quais as espécies selecionadas para elaboração das equações alométricas e quais as consequeências dessas escolhas. Saber a distribuição do diâmetro e da altura, me ajuda a identificar se as condições do estudo se assemelham a realidade de campo e, consequentemente, se devo ou não considerar as equações de outro estudo em área semelhante a área de projeto.

In [ ]:
df.isna().sum()

NUMERO_ARVORE    0
SUB_PARCELA      0
ID_parcela       0
FUSTE            0
DAP_cm           0
ALTURA_m         0
STATUS           0
NOME_POPULAR     0
GRUPO            0
dtype: int64


As métricas abaixo estão de acordo com o descrito no artigo.<br> Árvores de perfil baixo, raramente atingindo mais que 20 m de altura. 

In [100]:
print(f'Altura média: {df['ALTURA_m'].mean():.2f} m')
print(f'Altura mínima: {df['ALTURA_m'].min():.2f} m')
print(f'Altura máxima: {df['ALTURA_m'].max():.2f} m')
print('\n')
print(f'DAP médio: {df['DAP_cm'].mean():.2f} cm')
print(f'DAP mínimo: {df['DAP_cm'].min()} cm')
print(f'DAP máximo: {df['DAP_cm'].max()} cm')

Altura média: 5.59 m
Altura mínima: 1.00 m
Altura máxima: 14.20 m


DAP médio: 7.41 cm
DAP mínimo: 3.0 cm
DAP máximo: 42.0 cm


Extrapolar a equação para indivíduos com DAP > 30 cm não é recomendado, pois mesmo que sejam poucos indivíduos, podem introduzir grandes erros na estimativa. Ao todo, existem 13 indivíduos com DAP >= 30 cm, os quais serão removidos do dataset principal.

In [103]:
print(f'Indivíduos com DAP >= 30 cm removidos: {(df['DAP_cm'] >= 30).sum()}')

# Removendo os indivíduos com DAP >= 30 cm
df = df[df['DAP_cm'] <= 29]

Indivíduos com DAP >= 30 cm removidos: 0


Alguns gráficos exploratórios.

In [ ]:
plt.scatter(df['DAP_cm'], df['ALTURA_m'],
            alpha = 0.3,
            s = 20) # s = size
plt.xlabel('DAP (cm)')
plt.ylabel('Altura (m)')
plt.title('Distribuição DAP x Altura')
plt.show()

In [ ]:
plt.hist(df['DAP_cm'])
plt.xlabel('DAP (cm)')
plt.ylabel('Ocorrências')
plt.title('Distribuição dos diâmetros')
plt.show()

In [ ]:
plt.hist(df['ALTURA_m'])
plt.xlabel('Altura (m)')
plt.ylabel('Ocorrências')
plt.title('Distribuição das alturas')
plt.show()

Como o artigo possui equações específicas para algumas espécies, pensei em aplicá-las nas espécies identificadas no dataset, entretanto, uma vez que há apenas a o nome comum (que pode sofrer variações regionais), optei por aplicar apenas a equação generalista. Segundo o artigo, a eq generalista é ideal para estimar biomassa no bioma, enquanto equações específicas são mais indicadas para estudos populacionais. 

In [ ]:
individuos_unicos = df['NOME_POPULAR'].value_counts()
print(individuos_unicos.to_string())

# Aplicar equações específicas para espécies que aparecem no artigo
# Espécie           Nome Comum      Ocorrências
# C. sonderianus = Marmeleiro (~preto / branco), 1372 
# C. jamacaru = Mandacaru, 26 
# A. pyrifolium = Pereiro, 77 
# C. pyramidalis = Catingueira, 915 
# M. hostilis = Jurema preta, 180 
# M. urundeuva = Aroeira, 19 

# Processando os dados

As variáveis apresentadas nas equações desevolvidas por Sampaio & Silva e utilizadas neste script estão nas seguintes unidades:
- Área basal a altura do peito (ABH): cm²
- Diâmetro a altura do peito (DBH): cm
- Altura (H): m

Como uma das equações utiliza área basal como dado de entrada, vamos criar uma coluna apenas com essa variável, expressa em cm².

In [104]:
df['AREA_BASAL_cm2'] = (pi/4) * pow(df['DAP_cm'], 2)

O desafio pede estoque de biomassa em Mg ha e em área total. Nesse ponto fiquei em dúvida se devo ou não incluir o estoque em na biomassa morta. No contexto de projetos de carbono, um dos pools refere-se justamente da _deadwood_, mas normalmente, se calcula biomassa viva (contexto de inventários comum com nativas / plantadas).

A princípio, irei excluir e calcular a estimativa separada, assim como análise da variância, erro amostral e intervalo de confiança. 

In [105]:
# Pool deadwood faz sentido ser incluído?
df = df[df['STATUS'] == 'viva']

## Equação simples entrada
### Aplicação da equação geral exceto para indivíduos de _C. jamacaru_

In [130]:
df_simples = df

df_simples['BIOMASSA_kg'] = df.apply(lambda x: eq_biomassa_jamacaru(x['DAP_cm'])
                                    if x['NOME_POPULAR'] == 'Mandacaru'
                                    else eq_geral_simples_entrada(x['AREA_BASAL_cm2']),
                                    axis=1)
df_simples

,NUMERO_ARVORE,SUB_PARCELA,ID_parcela,FUSTE,DAP_cm,ALTURA_m,STATUS,NOME_POPULAR,GRUPO,AREA_BASAL_cm2,BIOMASS_kg,BIOMASSA_kg,BIOMASSA_t
0,rg01,DAP_3_Sub,P6_S15,a,8.1,6.0,viva,pacote,Cristalino,51.529974,21.042371,21.042371,0.021042
1,rg02,DAP_3_Sub,P6_S15,a,3.0,6.0,viva,mororo,Cristalino,7.068583,2.153351,2.153351,0.002153
2,rg03,DAP_3_Sub,P6_S15,a,6.8,6.0,viva,sabia,Cristalino,36.316811,14.084113,14.084113,0.014084
3,rg04,DAP_3_Sub,P6_S15,a,8.4,6.0,viva,pacote,Cristalino,55.417694,22.874021,22.874021,0.022874
4,rg05,DAP_3_Sub,P6_S15,a,6.0,6.5,viva,sabia,Cristalino,28.274334,10.567656,10.567656,0.010568
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9318,rg17,DAP_3_Sub,P1_S1,a,3.2,3.8,viva,mororo,Sedimentar,8.042477,2.497128,2.497128,0.002497
9319,rg18,DAP_3_Sub,P1_S1,a,4.3,3.8,viva,mororo,Sedimentar,14.522012,4.919621,4.919621,0.004920
9320,rg19,DAP_3_Sub,P1_S1,a,3.8,2.5,viva,angico branco,Sedimentar,11.341149,3.704460,3.704460,0.003704
9321,rg20,DAP_3_Sub,P1_S1,a,3.1,3.0,viva,sp18,Sedimentar,7.547676,2.321649,2.321649,0.002322


A equação retorna o dado em kg, portanto, vamos criar uma coluna para toneladas.

In [131]:
df_simples['BIOMASSA_t'] = df_simples['BIOMASSA_kg'] / 1000

Como as parcelas possuem sub parcelas, é preciso calcular um fator de expansão para cada área. 
- Parcela principal >> 20x50 m = 1000 m²
- Sub parcelas (2 por parcela) >> 10x10 m * 2 = 200 m² 

Um subset com cada conjunto facilita a aplicação dos fatores corretos. 

In [132]:
df_parcela = df_simples[df_simples['SUB_PARCELA'] == 'DAP_10_Plot']
df_sub = df_simples[df_simples['SUB_PARCELA'] == 'DAP_3_Sub']

area_parcela_m2 = 1000
area_parcela_ha = area_parcela_m2 / 10_000
print(f'Área parcela DAP > 10 cm: {area_parcela_m2} m²')

area_sub_m2 = 200
area_sub_ha = area_sub_m2 / 10_000
print(f'Área parcela DAP > 3 cm: {area_sub_m2} m²')

fator_exp_parcela = 10_000 / area_parcela_m2
print(f'Fator de expansão para a parcela: {fator_exp_parcela} ha^(-1)')

fator_exp_sub = 10_000 / area_sub_m2
print(f'Fator de expansão para a parcela: {fator_exp_sub} ha^(-1)')

Área parcela DAP > 10 cm: 1000 m²
Área parcela DAP > 3 cm: 200 m²
Fator de expansão para a parcela: 10.0 ha^(-1)
Fator de expansão para a parcela: 50.0 ha^(-1)


De posse dos fatores de expansão, os dados são agrupados pelo ID de cada parcela e realizado o produto da biomassa (t) calculada com o fator de expansão. O resultado de cada registro de uma parcela _i_ é somado obtendo o total por parcela.

In [ ]:
biomassa_parcela = df_parcela.groupby('ID_parcela')['BIOMASSA_t'].sum() * fator_exp_parcela # resultado em t/ha ou Mg/ha
biomassa_sub = df_sub.groupby('ID_parcela')['BIOMASSA_t'].sum() * fator_exp_sub 
biomassa_total_t_ha = (biomassa_parcela + biomassa_sub).to_frame('BIOMASSA_total_t_ha')

ID_parcela
P1_S1     37.994914
P1_S10    48.295173
P1_S11    20.649378
P1_S12    24.384795
P1_S13    16.160468
            ...    
P6_S5     27.007294
P6_S6     36.324818
P6_S7     26.361427
P6_S8      4.598667
P6_S9     39.493887
Name: BIOMASSA_t, Length: 78, dtype: float64
ID_parcela
P1_S1      5.620440
P1_S10    28.229421
P1_S11    27.439587
P1_S12    20.141452
P1_S13    38.511903
            ...    
P6_S5     39.685768
P6_S6     27.973926
P6_S7     43.220827
P6_S8     15.490175
P6_S9     25.556497
Name: BIOMASSA_t, Length: 78, dtype: float64


Unindo novamente o subset ao dataset principal e sumarisando os dados 

In [134]:
df_simples = df_simples.merge(biomassa_total_t_ha, how='left', on='ID_parcela')

In [ ]:
media_biomassa_t_ha = df_simples['BIOMASSA_total_t_ha'].mean()
std_biomassa_t_ha = df_simples['BIOMASSA_total_t_ha'].std()
n = df_simples['BIOMASSA_total_t_ha'].count()
erro_padrao = std_biomassa_t_ha/m.sqrt(n)



-- Biomassa --
Média: 49.29 Mt/ha
Desvio padrão da média: 13.56
Indivíduos avaliados: 8387


In [ ]:
t = stats.t.ppf(q=0.975, df= n-1) # t de student para 95#
ic = t * erro_padrao # Intervalo de confiança
erro_amostral = (ic / media_biomassa_t_ha) * 100



Média: 49.29 t/ha
IC (95%): +/- 0.29 t/ha
Erro amostral: 0.6%


In [159]:
resultados_simples(media_biomassa_t_ha, std_biomassa_t_ha, n, ic, erro_amostral, erro_padrao, area_total_ha)

--- Resultados ---
Estoque de biomassa: 49.29 Mt/ha
Área total: 6193.34
Estoque de biomassa em área total: 305,292.37 Mg
Desvio padrão da média: 13.56
Indivíduos avaliados: 8387
IC (95%): +/- 0.29 Mg/ha
Erro padrão: 0.15 Mg/ha
Erro amostral: 0.6%

